In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week7-lesson-2"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
##! hadoop fs -head /public/trendytech/datasets/cust_transf.csv

## Cache and Persist

In [3]:
order = 'order_id long , order_date date , cust_id long, status string'

In [4]:
df1 = spark.read.schema(order).option("header","false").csv('/public/trendytech/orders/orders_1gb.csv')

In [5]:
df1.cache()

order_id,order_date,cust_id,status
1,2013-07-25,11599,CLOSED
2,2013-07-25,256,PENDING_PAYMENT
3,2013-07-25,12111,COMPLETE
4,2013-07-25,8827,CLOSED
5,2013-07-25,11318,COMPLETE
6,2013-07-25,7130,COMPLETE
7,2013-07-25,4530,COMPLETE
8,2013-07-25,2911,PROCESSING
9,2013-07-25,5657,PENDING_PAYMENT
10,2013-07-25,5648,PENDING_PAYMENT


In [6]:
df2=df1.filter("cust_id = 1119198")

In [7]:
df2.show() ## this calls for the caching , runs slow

+--------+----------+-------+------+
|order_id|order_date|cust_id|status|
+--------+----------+-------+------+
+--------+----------+-------+------+



In [8]:
df2.show()  ## this utilises the cached data. runs fast

+--------+----------+-------+------+
|order_id|order_date|cust_id|status|
+--------+----------+-------+------+
+--------+----------+-------+------+



In [9]:
df1.unpersist()

order_id,order_date,cust_id,status
1,2013-07-25,11599,CLOSED
2,2013-07-25,256,PENDING_PAYMENT
3,2013-07-25,12111,COMPLETE
4,2013-07-25,8827,CLOSED
5,2013-07-25,11318,COMPLETE
6,2013-07-25,7130,COMPLETE
7,2013-07-25,4530,COMPLETE
8,2013-07-25,2911,PROCESSING
9,2013-07-25,5657,PENDING_PAYMENT
10,2013-07-25,5648,PENDING_PAYMENT


In [10]:
df3 = df1.select("order_id","status").filter("status == 'CLOSED'").cache()

In [11]:
df3.show() ## this initiates the caching, but since its only a show() just 1 paritiion is cached. because only that much is required for show()

+--------+------+
|order_id|status|
+--------+------+
|       1|CLOSED|
|       4|CLOSED|
|      12|CLOSED|
|      18|CLOSED|
|      24|CLOSED|
|      25|CLOSED|
|      37|CLOSED|
|      51|CLOSED|
|      57|CLOSED|
|      61|CLOSED|
|      62|CLOSED|
|      87|CLOSED|
|      90|CLOSED|
|     101|CLOSED|
|     116|CLOSED|
|     129|CLOSED|
|     133|CLOSED|
|     191|CLOSED|
|     201|CLOSED|
|     211|CLOSED|
+--------+------+
only showing top 20 rows



In [12]:
df3.count() ## this will do the caching for all partitions now, as whole data is required to compute this. but runs slow as cache is being done

2833500

In [13]:
df3.count() ## this utlizes the entire cache and runs fast

2833500

In [14]:
df1.select("order_id","status").filter("status == 'CLOSED'").count() ## this one utilizes cache

2833500

In [15]:
df1.filter("status == 'CLOSED'").select("order_id","status").count()  
## this wont hit cache eventhough its the same transformation due the spark 'analysed plan being different' from the previous transformation

2833500

In [16]:
df4 = df1.select("order_id","status").filter("order_id > 10").cache() 

In [17]:
df4.count()

25827375

In [18]:
df4.count()

25827375

In [19]:
df1.select("order_id","status").filter("order_id > 100").count()
## this also wont hit cache, even though '> 100' means records from  '> 10' cache is included. due to diff in analyzed plan

25793625

In [20]:
df4.select("order_id","status").filter("order_id > 10").count() ## but this will hit the cache as it operates on df4

25827375

In [21]:
df4.select("order_id","status").filter("order_id > 100").count()

25793625

In [ ]:
df1.select("status").filter("order_id > 10").cache() ## This also will not hit the cache 

In [ ]:
df1.select("order_id").filter("order_id > 10").cache() ## This will hit the cache as the order of cols in select is same as that in cache